In [ ]:
file_name = ""
tiled_id = ""

# Panda Scan Analysis Notebook

In [ ]:
# imports:
import numpy as np
import json
import matplotlib.pyplot as plt
import h5py
from scipy.signal import savgol_filter

In [ ]:
# artifacts:
ARTIFACTS = {
    "notebook": "/tmp/notebook.html",
    "plot": "/tmp/plot_result.png",
    "pca_plot": "/tmp/plot_pca.png",
    "report": "/tmp/report.json",
    }

In [ ]:
# read dataset and get (x, y) data:
fpath = f"/data/{file_name}.nxs"

instrument_path = "/entry/instrument/panda1"
xname = "inenc-1-val_dataset"
yname = "fmc_in-val1_dataset"

meta_path = "/entry/plan_metadata/plan_args"

with h5py.File(fpath, "r") as f:

    # panda1 data
    x = f[f"{instrument_path}/{xname}"][...]
    y = f[f"{instrument_path}/{yname}"][...]

    # plan metadata
    npos = f[f"{meta_path}/capture_positions"].len()
    nrep = f[f"{meta_path}/num_seqtable_repeats"].asstr("utf-8")[()]
    ntraj = f[f"{meta_path}/num_trajectory_points"].asstr("utf-8")[()]
    nsweep = f[f"{meta_path}/number_of_sweeps"].asstr("utf-8")[()]
    nstep = f[f"{meta_path}/stepsize"].asstr("utf-8")[()]

In [ ]:
nrep = int(nrep); ntraj = int(ntraj); nsweep = int(nsweep); nstep = float(nstep)

ncol = int(y.shape[0]/nrep)

In [ ]:
y1 = y.copy()
y1 = y1.reshape((nrep, ncol))
x = x.reshape((nrep, ncol))

y1[1::2,:] = y1[1::2,::-1]

## Utils for scan diagnostics:

In [ ]:
def repetition_processing(y:np.ndarray)->np.ndarray:
    """
    detect number of repetitions of scan.
    """
    print(f"Found {nrep} repetitions.")

    noise = np.zeros(nrep)
    std = np.zeros(nrep)
    avg = np.zeros_like(y)

    for i in range(nrep):

        yf = y[:i+1,:].mean(axis=0)
        sg = savgol_filter(yf, window_length=int(len(y)/10),
                           polyorder=2)
        noise_out = np.std(yf - sg)
        noise[i] = noise_out
        std[i] = np.std(yf)
        avg[i,:] = yf
    
    return noise, avg, std

## Run Analysis:

<h4>Noise:</h4>

In [ ]:
noise, avg, std = repetition_processing(y1)

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(10, 5), layout="tight")

xplot = np.linspace(1, len(noise)+1, len(noise))

fig.suptitle(f"Data for: {fpath.split('/')[-1]}\ntiled id: {tiled_id}")

ax[0,0].set_title("xy data")
ax[0,0].plot(x.T, y1.T, lw=1)

ax[0,1].set_title(r"$\frac{noise}{\sqrt{N}}$")
ax[0,1].plot(xplot, noise/np.sqrt(xplot), "o-")
ax[0,1].set_xlabel("Repetition No.")

ax[1,0].set_title("Mean XY")
ax[1,0].plot(x[0,:], avg.T)

ax[1,1].set_title(r"$\sigma^2$")
ax[1,1].plot(xplot, std, "x-")

[a.set_xlabel(xname) for a in [ax[0,0], ax[1,0]]]
[a.set_ylabel(yname) for a in [ax[0,0], ax[1,0]]]

[a.set_xlabel("Repetition Number") for a in [ax[0,1], ax[1,1]]]

fig.savefig(ARTIFACTS["plot"])
plt.show();

<h4>PCA Decomposition:</h4>

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize

In [ ]:
y2 = np.cumsum(y1, axis=0)
y2 = normalize(y2)

In [ ]:
ncomp = 5
model = PCA(n_components=ncomp)

In [ ]:
W = model.fit_transform(y2.T)
H = model.components_

In [ ]:
w_ax = [f"{i}_comp" for i in range(ncomp)]
p_ax = [f"{i}_pc" for i in range(ncomp)]
s_ax = ["data"]*int(ncomp/2)
s_ax = s_ax + (["scree"]*int(ncomp-int(ncomp/2)))

fig, ax = plt.subplot_mosaic([s_ax, p_ax, w_ax], figsize=(10, 8), layout="constrained")
ax["scree"].set_title("Eigenvalues")
ax["data"].set_title("Data")
ax["scree"].plot(np.arange(1, ncomp+1, 1), model.explained_variance_, "x-")
ax["data"].plot(x.T, y2.T)

for i in range(ncomp):
    ax[f"{i}_pc"].set_title(f"N={i+1}")
    ax[f"{i}_comp"].plot(np.arange(1, H.shape[1]+1), H[i,:])
    ax[f"{i}_pc"].plot(x[i,:], W[:,i])
    ax[f"{i}_comp"].set_title(f"N={i+1}")

ax["0_pc"].set_title("Components\n PC=1")

ax["0_comp"].set_title("Weights\n N=1")

fig.savefig(ARTIFACTS["pca_plot"])

plt.show();

<h4>Saving Data:</h4>

In [ ]:
report = {"snr": noise,
          "avg": avg,
          "std": std,
          "x": x[0,:]
          }

with open(ARTIFACTS["report"], "w") as fp:
    json.dump(report, fp, indent=2, default=str)
